In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="st_paraphrase_minilm_l6_cosine_threshold_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

label_names = ["not_paraphrase", "paraphrase"]



---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [4]:
model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"

model = SentenceTransformer(model_name, device=str(device))
print(model_name)



---[ TableVault Record ]---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/paraphrase-MiniLM-L6-v2
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
---[ TableVault Record ]---



In [6]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())



---[ TableVault Record ]---
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [7]:
batch_size = 128
threshold = 0.78

emb1_batches = []
emb2_batches = []

for i in tqdm(range(0, len(ds), batch_size)):
    batch_s1 = sent1[i:i + batch_size]
    batch_s2 = sent2[i:i + batch_size]

    e1 = model.encode(
        batch_s1,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    e2 = model.encode(
        batch_s2,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    emb1_batches.append(e1)
    emb2_batches.append(e2)

emb1 = torch.cat(emb1_batches, dim=0)
emb2 = torch.cat(emb2_batches, dim=0)

scores = (emb1 * emb2).sum(dim=-1)
y_pred = (scores >= threshold).long().cpu().numpy()
scores_np = scores.cpu().numpy()

print("done")



---[ TableVault Record ]---


  0%|          | 0/4 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [8]:

vault.create_record_list("minilm_cosine_threshold_prediction", column_names=["prediction", "score"])

for i in range(len(y_pred)):
    vault.append_record("minilm_cosine_threshold_prediction", 
                        {
                            "prediction": int(y_pred[i]),
                            "score": float(scores_np[i]) ,
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Row-level prediction dataset for the GLUE MRPC validation set produced by the sentence-transformers/paraphrase-MiniLM-L6-v2 model. Each record corresponds to one sentence pair from glue_mrpc_validation and stores two fields: prediction (binary label, where 0 = not_paraphrase and 1 = paraphrase) and score (cosine similarity between the normalized embeddings of sentence1 and sentence2). Predictions are generated by applying a fixed threshold of 0.78 to the cosine similarity score. In this workflow, this dataset serves as the per-example model output used for downstream evaluation, error analysis, and creation of the summary metrics dataset."
embedding = get_embeddings(description)
vault.create_description("minilm_cosine_threshold_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "model predictions", "model": "sentence-transformers/paraphrase-MiniLM-L6-v2", "method": "cosine similarity thresholding", "threshold": "0.78", "source": "glue/mrpc", "source_table": "glue_mrpc_validation", "split": "validation", "size": "408", "input_fields": "sentence1,sentence2", "output_fields": "prediction,score", "label_space": "not_paraphrase,paraphrase", "modality": "text-text", "metric_score": "cosine similarity", "embedding_normalization": "true"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("minilm_cosine_threshold_prediction", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=label_names))



---[ TableVault Record ]---
{'accuracy': 0.7009803921568627, 'f1': 0.7706766917293233, 'threshold': 0.78}
                precision    recall  f1-score   support

not_paraphrase       0.52      0.63      0.57       129
    paraphrase       0.81      0.73      0.77       279

      accuracy                           0.70       408
     macro avg       0.67      0.68      0.67       408
  weighted avg       0.72      0.70      0.71       408

---[ TableVault Record ]---



In [10]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", label_names[int(y_pred[i])])



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9130104184150696
true: 1 pred: 1 label: paraphrase
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.2962682843208313
true: 0 pred: 0 label: not_paraphrase
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.8564640283584595
true: 0 pred: 1 label: paraphrase
sentence1: The AFL-CIO is waiting until October to decide if it

In [11]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))



---[ TableVault Record ]---
num_errors: 122
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.8564640283584595
true: 0 pred: 1
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
score: 0.7582171559333801
true: 1 pred: 0
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
score: 0.5897153615951538
true: 1 pred: 0
idx: 6
sentence1: While dioxin levels in the environment were u

In [12]:

vault.create_record_list("st_paraphrase_minilm_l6_cosine_threshold_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("st_paraphrase_minilm_l6_cosine_threshold_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "minilm_cosine_threshold_prediction": [0, len(ds)]
                    })

summary

description = "This dataset stores the evaluation summary for the paraphrase detection workflow using sentence-transformers/paraphrase-MiniLM-L6-v2 on the GLUE MRPC validation set with cosine-similarity thresholding at 0.78. It contains a single summary record with three fields: accuracy (float), f1 (float), and classification_report (string). The metrics are computed by comparing thresholded cosine-similarity predictions from the minilm_cosine_threshold_prediction dataset against the ground-truth labels in glue_mrpc_validation. Its role in this workflow is to provide a compact, experiment-level performance summary for the notebook, linking the full validation set and per-example predictions to aggregate evaluation results."
embedding = get_embeddings(description)
vault.create_description("st_paraphrase_minilm_l6_cosine_threshold_mrpc_summary", description, embedding)

properties = {"dataset_type": "evaluation summary", "task": "paraphrase detection", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "sentence-transformers/paraphrase-MiniLM-L6-v2", "method": "cosine similarity thresholding", "threshold": "0.78", "input_type": "sentence pairs", "metrics": "accuracy,f1,classification_report", "label_space": "not_paraphrase,paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_paraphrase_minilm_l6_cosine_threshold_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [13]:
description = "This dataset documents a paraphrase detection experiment on the GLUE MRPC validation split using the sentence-transformers/paraphrase-MiniLM-L6-v2 model with cosine-similarity thresholding. It uses the input records from glue_mrpc_validation, where each example contains sentence1, sentence2, and label. The workflow generates a prediction table, minilm_cosine_threshold_prediction, with one record per input pair and fields prediction and score, where score is the cosine similarity between the normalized sentence embeddings and prediction is the binary paraphrase decision produced with threshold 0.78. It also generates st_paraphrase_minilm_l6_cosine_threshold_mrpc_summary, a summary table with fields accuracy, f1, and classification_report computed over the full validation set. In this workflow, st_paraphrase_minilm_l6_cosine_threshold_mrpc serves as the top-level experiment dataset linking the MRPC validation inputs, per-example model outputs, and aggregate evaluation results." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("st_paraphrase_minilm_l6_cosine_threshold_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "source": "glue/mrpc", "split": "validation", "size": "408", "domain": "news", "model": "sentence-transformers/paraphrase-MiniLM-L6-v2", "method": "cosine similarity thresholding", "threshold": "0.78", "labels": "binary", "output": "predictions and evaluation summary"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_paraphrase_minilm_l6_cosine_threshold_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

